# 08 — Workforce Forecasting
**Goal**: Forecast headcount needs, identify retirement risk

**ML Progression**: Headcount trends → Kaplan-Meier Survival → ARIMA

**HR Value**: Strategic workforce planning, hiring budget

**Employee Value**: Job security context, retirement planning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
from lifelines import KaplanMeierFitter
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/08_forecast'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(PROJECT_ROOT / 'data/raw/employee_data.csv')
df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Active employees: {len(df)}')

## 1. Age & Retirement Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['age'].dropna(), bins=25)
axes[0].set_title('Age Distribution of Active Employees')
df['retirement_risk'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Retirement Risk (Age >= 55)')
axes[1].set_xticklabels(['Low Risk', 'High Risk'], rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/08_age_retirement.png', bbox_inches='tight')
plt.show()

print(f'Employees at retirement risk (age >= 55): {df["retirement_risk"].sum()}')
print(f'Percentage: {df["retirement_risk"].mean() * 100:.1f}%')

## 2. Kaplan-Meier Survival Analysis

In [ ]:
# Prepare survival data from raw
surv_df = raw.copy()
surv_df['is_terminated'] = surv_df['EmployeeStatus'].str.lower().str.contains('terminat').astype(int)
start = pd.to_datetime(surv_df['StartDate'], format='mixed', errors='coerce')
exit = pd.to_datetime(surv_df['ExitDate'], format='mixed', errors='coerce')
today = pd.Timestamp.now()
surv_df['T'] = (exit.fillna(today) - start).dt.days
surv_df['E'] = surv_df['is_terminated']
surv_df = surv_df.dropna(subset=['T'])

kmf = KaplanMeierFitter()
kmf.fit(surv_df['T'], event_observed=surv_df['E'])

fig, ax = plt.subplots(figsize=(10, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title('Kaplan-Meier Employee Survival Curve')
ax.set_xlabel('Tenure (days)')
ax.set_ylabel('Survival Probability')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/08_km_survival.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Survival Analysis:')
print(f'  Median survival time: {kmf.median_survival_time_:.0f} days ({kmf.median_survival_time_/365.25:.1f} years)')
print(f'  Events observed: {surv_df["E"].sum()}')

## 3. Headcount Forecasting (ARIMA)

In [ ]:
# Create monthly headcount series
raw['start_dt'] = pd.to_datetime(raw['StartDate'], format='mixed', errors='coerce')
raw['exit_dt'] = pd.to_datetime(raw['ExitDate'], format='mixed', errors='coerce')

monthly = pd.date_range(start=raw['start_dt'].min(), end=today, freq='MS')
headcount = []
for m in monthly:
    started_before = raw['start_dt'] <= m
    not_exited = (raw['exit_dt'].isna()) | (raw['exit_dt'] > m)
    hc = (started_before & not_exited).sum()
    headcount.append(hc)

hc_series = pd.Series(headcount, index=monthly)
print(f'Headcount from {monthly[0].date()} to {monthly[-1].date()}')
print(f'  Current headcount: {headcount[-1]}')

In [ ]:
try:
    model = ARIMA(hc_series, order=(2, 1, 2))
    fitted = model.fit()
    forecast = fitted.forecast(steps=12)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(hc_series.index, hc_series, label='Historical', color='steelblue')
    ax.plot(forecast.index, forecast, label='Forecast (12 months)', color='red', linestyle='--')
    ax.set_title('Headcount Forecast (ARIMA)')
    ax.set_ylabel('Active Employees')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/08_headcount_forecast.png', bbox_inches='tight')
    plt.show()
    print(f'\nForecasted next 12 months:\n{forecast.round(0).astype(int).to_string()}')
except Exception as e:
    print(f'ARIMA failed (not enough data points?): {e}')

## 4. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Retirement risk: {df["retirement_risk"].sum():.0f} employees (>= 55 years old)')
print(f'2. Median tenure survival: {kmf.median_survival_time_/365.25:.1f} years')
print(f'3. Current headcount: {headcount[-1]}')
print()
print('--- HR Action Items ---')
print('- Develop succession plans for retirement-risk employees')
print('- Use survival curves to predict future hiring needs')
print()
print('--- Employee Impact ---')
print('- Retirement planning visibility')
print('- Better workforce stability through proactive planning')